# `SCEN_TASK_NO_Insert_TRG_EMP.sql` Conversion

**Conversion Timestamp:** 2024-01-01 00:00:00

**Description:** This notebook loads data into `workspace.hr.trg_emp` from `workspace.hr.employees` using a direct INSERT statement.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type (e.g., F for Full, I for Incremental)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "100", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT CAST(${DATASOURCE_NUM_ID} AS BIGINT) AS datasource_num_id;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT CAST(${ETL_PROC_WID} AS BIGINT) AS etl_proc_wid;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT CAST(${ODI_SESS_NO} AS BIGINT) AS odi_sess_no;

In [ ]:
print("Displaying ETL Parameter Values:")
display(spark.sql("SELECT * FROM v_etl_job_type"))
display(spark.sql("SELECT * FROM v_datasource_num_id"))
display(spark.sql("SELECT * FROM v_etl_proc_wid"))
display(spark.sql("SELECT * FROM v_odi_sess_no"))

## Target Table Load

In [ ]:
%sql
-- SCEN_TASK_NO in {10}: Initial markers (no SQL action)
-- SCEN_TASK_NO in {20}: Initial markers (no SQL action)
-- SCEN_TASK_NO in {30}: Insert into target table
INSERT 
  INTO workspace.hr.trg_emp
  (
    employee_id ,
    first_name ,
    last_name ,
    email ,
    phone_number ,
    hire_date ,
    job_id ,
    salary ,
    commission_pct ,
    manager_id ,
    department_id 
  ) 
SELECT 
  employees.employee_id ,
  employees.first_name ,
  employees.last_name ,
  employees.email ,
  employees.phone_number ,
  employees.hire_date ,
  employees.job_id ,
  employees.salary ,
  employees.commission_pct ,
  employees.manager_id ,
  employees.department_id  
FROM 
  workspace.hr.employees employees;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_trg_emp
FROM workspace.hr.trg_emp;

## Conversion Notes and Manual Actions Required

- This script performs a direct full load `INSERT` from `workspace.hr.employees` to `workspace.hr.trg_emp`.
- Oracle `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable in Databricks Delta Lake.
- Schema `HR` has been mapped to `workspace.hr`.
- Table names `TRG_EMP` and `EMPLOYEES` have been converted to lowercase: `trg_emp` and `employees`.
- Ensure the target table `workspace.hr.trg_emp` exists and has the appropriate schema before running this notebook. If not, a `CREATE TABLE` DDL would be required (not provided in source ODI).